# Methylation Landscape — Exploratory Data Analysis

Preliminary EDA of **ONT native methylation** profiles (dorado duplex → `modkit pileup` → bedMethyl) across four *Sorghum bicolor* accessions, performed **before** DMR analysis.

| # | Section | Purpose |
|---|---------|---------|
| 1 | Average methylation per context | Confirm CG > CHG > CHH expected pattern per sample |
| 2 | Methylation at genomic features | Promoter vs gene body vs intergenic |
| 3 | PCA / hierarchical clustering | Global sample-level similarity |
| 4 | Coverage and methylation distribution | ONT-specific QC (M-bias analogue) |

**Samples:** SBC4 (TAA medium), SBC10 (TAA high), SBC11 (TAA low), SBC23 (TAA medium replicate)  
**Contexts available:** `CpG` (symmetric, `--cpg --combine-strands`) and `allC` (all cytosines, both strands)


In [2]:
import subprocess
import tempfile
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pyfaidx


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
WDIR        = Path('/home/daffa/Work/2026/02-JSPP67')
REF_FASTA   = WDIR / 'data/reference/asm_BTx623.fna'
REF_GFF     = WDIR / 'data/reference/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff'
RESULTS_DIR = WDIR / 'results/dmr_analysis'
OUT_DIR     = WDIR / 'results/methylation_landscape'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLES  = ['SBC4', 'SBC10', 'SBC11', 'SBC23']
CONTEXTS = ['CpG', 'allC']

# TAA phenotype metadata for labels / colours
SAMPLE_META = {
    'SBC4':  {'label': 'SBC4\n(TAA med)',  'color': '#4CAF50'},
    'SBC10': {'label': 'SBC10\n(TAA high)', 'color': '#2196F3'},
    'SBC11': {'label': 'SBC11\n(TAA low)',  'color': '#F44336'},
    'SBC23': {'label': 'SBC23\n(TAA med)',  'color': '#8BC34A'},
}

print('WDIR      :', WDIR)
print('REF_FASTA :', REF_FASTA,   '— exists:', REF_FASTA.exists())
print('REF_GFF   :', REF_GFF,     '— exists:', REF_GFF.exists())
print('OUT_DIR   :', OUT_DIR)


WDIR      : /home/daffa/Work/2026/02-JSPP67
REF_FASTA : /home/daffa/Work/2026/02-JSPP67/data/reference/asm_BTx623.fna — exists: True
REF_GFF   : /home/daffa/Work/2026/02-JSPP67/data/reference/GCF_000003195.3_Sorghum_bicolor_NCBIv3_genomic.gff — exists: True
OUT_DIR   : /home/daffa/Work/2026/02-JSPP67/results/methylation_landscape


In [4]:
# ── bedMethyl column names (18-col modkit pileup output) ─────────────────────
BEDMETHYL_COLS = [
    'chrom', 'start', 'end', 'mod_code', 'score', 'strand',
    'start2', 'end2', 'color',
    'N_valid_cov', 'pct_mod', 'N_mod',
    'N_canonical', 'N_other_mod', 'N_del', 'N_fail', 'N_diff', 'N_nocall',
]

# ── Data availability check ───────────────────────────────────────────────────
rows = []
for ctx in CONTEXTS:
    for s in SAMPLES:
        fpath = RESULTS_DIR / ctx / s / f'{s}.bedMethyl'
        rows.append({'context': ctx, 'sample': s, 'exists': fpath.exists(), 'path': str(fpath)})

avail_df = pd.DataFrame(rows)
pivot = avail_df.pivot(index='context', columns='sample', values='exists')
print('Data availability (True = bedMethyl present):')
print(pivot.to_string())
print()

# Dicts: context → list of available sample names
available = {
    ctx: [s for s in SAMPLES if (RESULTS_DIR / ctx / s / f'{s}.bedMethyl').exists()]
    for ctx in CONTEXTS
}
for ctx, samples in available.items():
    print(f'  {ctx}: {samples}')


Data availability (True = bedMethyl present):
sample   SBC10  SBC11  SBC23  SBC4
context                           
CpG       True   True   True  True
allC      True   True   True  True

  CpG: ['SBC4', 'SBC10', 'SBC11', 'SBC23']
  allC: ['SBC4', 'SBC10', 'SBC11', 'SBC23']


# Additional Section
Average number of CpG sites contained in the DMR

## Section 1 — Average Methylation by Context (CG / CHG / CHH)

Uses the `allC` bedMethyl (all cytosines, both strands retained).  
Cytosines are classified as **CG**, **CHG**, or **CHH** using the reference FASTA sequence context (`pyfaidx`):

| Strand | Context rule |
|--------|-------------|
| + | CG if `ref[p+1]=='G'`; CHG if `ref[p+2]=='G'`; else CHH |
| − | CG if `ref[p−1]=='C'`; CHG if `ref[p−2]=='C'`; else CHH |

**Weighted mean methylation** = Σ N_mod / Σ N_valid_cov × 100  (weights by coverage to avoid bias from low-coverage sites).


In [5]:
# ── Load allC bedMethyl for available samples ─────────────────────────────────
allC_data = {}
for sample in available['allC']:
    fpath = RESULTS_DIR / 'allC' / sample / f'{sample}.bedMethyl'
    print(f'  Loading {sample} ... ', end='', flush=True)
    df = pd.read_csv(
        fpath, sep='\t', header=None, names=BEDMETHYL_COLS,
        dtype={
            'chrom': str, 'start': int, 'end': int, 'strand': str,
            'N_valid_cov': int, 'pct_mod': float, 'N_mod': int,
        },
    )
    allC_data[sample] = df[['chrom', 'start', 'end', 'strand',
                              'N_valid_cov', 'pct_mod', 'N_mod']].copy()
    print(f'{len(df):,} rows  |  chroms: {df["chrom"].nunique()}')

if not allC_data:
    raise RuntimeError('No allC bedMethyl files found. Run the DMR pipeline first.')

# Quick peek
sample0 = list(allC_data.keys())[0]
print()
allC_data[sample0].head(3)


  Loading SBC4 ... 272,966,382 rows  |  chroms: 815
  Loading SBC10 ... 278,649,897 rows  |  chroms: 833
  Loading SBC11 ... 280,783,593 rows  |  chroms: 852
  Loading SBC23 ... 274,802,759 rows  |  chroms: 793



,chrom,start,end,strand,N_valid_cov,pct_mod,N_mod
0,NC_012870.2,145,146,-,13,0.00,0
1,NC_012870.2,147,148,-,13,0.00,0
2,NC_012870.2,154,155,+,17,5.88,1


In [ ]:
# ── Reference-based context classification ────────────────────────────────────

def classify_contexts(df, fasta):
    """
    Add 'context' column (CG / CHG / CHH) to a bedMethyl DataFrame.

    Vectorised per-chromosome using NumPy: the reference chromosome sequence is
    converted to a byte array and context is determined by comparing adjacent
    nucleotides in the appropriate direction for each strand.

    Parameters
    ----------
    df    : DataFrame with columns ['chrom','start','strand'] and a default
            RangeIndex (reset_index() applied before passing).
    fasta : pyfaidx.Fasta object built from the reference FASTA.
    """
    ctx_arr   = np.full(len(df), 'CHH', dtype='U3')
    avail_chr = set(fasta.keys())

    for chrom, grp in df.groupby('chrom', sort=False):
        if chrom not in avail_chr:
            continue
        seq  = np.frombuffer(fasta[chrom][:].seq.upper().encode('ascii'), dtype='S1')
        clen = len(seq)

        # ── + strand: look right (p+1, p+2) ──────────────────────────────────
        plus_idx = grp.index[grp['strand'] == '+'].values
        if len(plus_idx):
            p  = df.loc[plus_idx, 'start'].values
            n1 = seq[np.clip(p + 1, 0, clen - 1)]
            n2 = seq[np.clip(p + 2, 0, clen - 1)]
            ctx = np.where(n1 == b'G', 'CG', np.where(n2 == b'G', 'CHG', 'CHH'))
            ctx[p + 2 >= clen] = 'CHH'           # chromosome-end edge
            ctx_arr[plus_idx]  = ctx

        # ── − strand: look left (complement: p-1, p-2) ───────────────────────
        minus_idx = grp.index[grp['strand'] == '-'].values
        if len(minus_idx):
            p  = df.loc[minus_idx, 'start'].values
            m1 = seq[np.clip(p - 1, 0, clen - 1)]
            m2 = seq[np.clip(p - 2, 0, clen - 1)]
            ctx = np.where(m1 == b'C', 'CG', np.where(m2 == b'C', 'CHG', 'CHH'))
            ctx[p < 2] = 'CHH'                    # chromosome-start edge
            ctx_arr[minus_idx] = ctx

    return df.assign(context=ctx_arr)


# ── Load reference FASTA (pyfaidx — lazy-loaded per chromosome) ───────────────
print('Opening reference FASTA with pyfaidx ...')
genome = pyfaidx.Fasta(str(REF_FASTA), build_index=False)
print(f'  {len(genome.keys())} sequences in {REF_FASTA.name}')
print()

# ── Apply to each sample ──────────────────────────────────────────────────────
allC_ctx = {}
for sample, df in allC_data.items():
    print(f'  Classifying {sample} ({len(df):,} positions) ...', end=' ', flush=True)
    df_ri = df.reset_index(drop=True)          # ensure default RangeIndex
    allC_ctx[sample] = classify_contexts(df_ri, genome)
    vc = allC_ctx[sample]['context'].value_counts()
    print(f'CG: {vc.get("CG",0):,}  CHG: {vc.get("CHG",0):,}  CHH: {vc.get("CHH",0):,}')


In [ ]:
# ── Weighted mean methylation per context per sample ─────────────────────────
records = []
for sample, df in allC_ctx.items():
    for ctx, grp in df.groupby('context'):
        n_cov = grp['N_valid_cov'].sum()
        n_mod = grp['N_mod'].sum()
        wmean = (n_mod / n_cov * 100) if n_cov > 0 else 0.0
        records.append({'sample': sample, 'context': ctx, 'mean_meth_pct': wmean,
                        'n_sites': len(grp)})

ctx_summary = pd.DataFrame(records)
print(ctx_summary.pivot(index='sample', columns='context', values='mean_meth_pct')
                 .round(2).to_string())

# ── Grouped bar chart ─────────────────────────────────────────────────────────
CTX_ORDER    = ['CG', 'CHG', 'CHH']
SAMPLE_ORDER = [s for s in SAMPLES if s in allC_ctx]
CTX_COLORS   = {'CG': '#2196F3', 'CHG': '#FF9800', 'CHH': '#4CAF50'}

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(SAMPLE_ORDER))
w = 0.25

for i, ctx in enumerate(CTX_ORDER):
    vals = [
        ctx_summary.loc[
            (ctx_summary['sample'] == s) & (ctx_summary['context'] == ctx),
            'mean_meth_pct',
        ].values[0] if len(ctx_summary[
            (ctx_summary['sample'] == s) & (ctx_summary['context'] == ctx)
        ]) else 0
        for s in SAMPLE_ORDER
    ]
    offset = (i - 1) * w
    ax.bar(x + offset, vals, w, label=ctx, color=CTX_COLORS[ctx], edgecolor='white',
           linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([SAMPLE_META[s]['label'] for s in SAMPLE_ORDER])
ax.set_ylabel('Weighted mean methylation (%)')
ax.set_title('Average methylation by sequence context (allC bedMethyl)')
ax.legend(title='Context', frameon=False)
ax.set_ylim(0, 100)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_avg_methylation_by_context.pdf', bbox_inches='tight')
plt.show()


## Section 2 — Methylation at Genomic Features

Compares **CpG methylation** levels across three feature categories:

| Feature | Definition |
|---------|-----------|
| **Promoter** | 2 000 bp upstream of the TSS (strand-aware, clamped to 0) |
| **Gene body** | TSS → TES as annotated in the NCBI GFF3 |
| **Intergenic** | CpG positions not overlapping either feature |

Promoter takes priority over gene body at overlapping positions.  
Chromosome name check is performed before intersecting to catch annotation mismatches early.


In [ ]:
# ── Parse NCBI GFF3 → gene body + promoter BED files ─────────────────────────
PROMOTER_BP = 2000    # bp upstream of TSS

GFF_COLS = ['seqid', 'source', 'type', 'start', 'end',
            'score', 'strand', 'phase', 'attributes']
gff = pd.read_csv(REF_GFF, sep='\t', comment='#', header=None,
                  names=GFF_COLS, dtype={'start': int, 'end': int})
genes = gff[gff['type'] == 'gene'].copy()
print(f'Loaded {len(genes):,} gene features from GFF3')

# Gene body BED (0-based half-open)
gene_bed = pd.DataFrame({
    'chrom':  genes['seqid'].values,
    'start':  genes['start'].values - 1,    # GFF 1-based → BED 0-based
    'end':    genes['end'].values,
    'name':   'gene_body',
    'score':  0,
    'strand': genes['strand'].values,
})

# Promoter BED (strand-aware, 2 kb upstream of TSS)
plus_mask  = genes['strand'].values == '+'
tss        = np.where(plus_mask, genes['start'].values - 1, genes['end'].values)
promo_start = np.where(plus_mask, np.maximum(0, tss - PROMOTER_BP), tss)
promo_end   = np.where(plus_mask, tss,                              tss + PROMOTER_BP)

promoter_bed = pd.DataFrame({
    'chrom':  genes['seqid'].values,
    'start':  promo_start,
    'end':    promo_end,
    'name':   'promoter',
    'score':  0,
    'strand': genes['strand'].values,
})
promoter_bed = promoter_bed[promoter_bed['start'] < promoter_bed['end']].copy()

# ── Chromosome name compatibility check ───────────────────────────────────────
_cpg_sample = available['CpG'][0] if available['CpG'] else None
if _cpg_sample:
    _peek = pd.read_csv(
        RESULTS_DIR / 'CpG' / _cpg_sample / f'{_cpg_sample}.bedMethyl',
        sep='\t', header=None, usecols=[0], nrows=2000, names=['chrom'],
    )
    bm_chroms  = set(_peek['chrom'].unique())
    gff_chroms = set(gene_bed['chrom'].unique())
    shared     = bm_chroms & gff_chroms
    print(f'\nbedMethyl chroms (sample): {sorted(bm_chroms)[:4]}')
    print(f'GFF3 chroms (sample):      {sorted(gff_chroms)[:4]}')
    print(f'Shared: {len(shared)} chromosomes — OK' if shared
          else 'WARNING: No shared chromosomes — check name mapping!')

# ── Write temp BED files ──────────────────────────────────────────────────────
TMPDIR = Path(tempfile.mkdtemp(prefix='methyl_eda_'))
gene_bed_path     = TMPDIR / 'genes.bed'
promoter_bed_path = TMPDIR / 'promoters.bed'

gene_bed.sort_values(['chrom', 'start']).to_csv(
    gene_bed_path, sep='\t', header=False, index=False)
promoter_bed.sort_values(['chrom', 'start']).to_csv(
    promoter_bed_path, sep='\t', header=False, index=False)

print(f'\nGene body BED:  {gene_bed_path}  ({len(gene_bed):,} features)')
print(f'Promoter BED:   {promoter_bed_path}  ({len(promoter_bed):,} features)')


In [ ]:
# ── Feature-type assignment via bedtools intersect ────────────────────────────

def assign_feature_type(sample, gene_bed_path, promoter_bed_path, tmpdir):
    """
    Load the CpG bedMethyl for *sample*, label each position as
    'promoter' | 'gene_body' | 'intergenic' using bedtools intersect,
    and return the annotated DataFrame.

    Promoter takes priority over gene body for positions that overlap both.
    """
    fpath = RESULTS_DIR / 'CpG' / sample / f'{sample}.bedMethyl'
    bm = pd.read_csv(
        fpath, sep='\t', header=None, names=BEDMETHYL_COLS,
        usecols=[0, 1, 2, 9, 10],
        dtype={'chrom': str, 'start': int, 'end': int,
               'N_valid_cov': int, 'pct_mod': float},
    )
    bm.columns = ['chrom', 'start', 'end', 'N_valid_cov', 'pct_mod']
    bm = bm.sort_values(['chrom', 'start']).reset_index(drop=True)
    bm['row_idx'] = bm.index          # integer row tag preserved through bedtools

    tmp_bed = tmpdir / f'{sample}_cpg.tmp.bed'
    bm[['chrom', 'start', 'end', 'row_idx']].to_csv(
        tmp_bed, sep='\t', header=False, index=False)

    def intersect_row_indices(feat_path):
        res = subprocess.run(
            ['bedtools', 'intersect', '-a', str(tmp_bed),
             '-b', str(feat_path), '-u'],
            capture_output=True, text=True, check=True,
        )
        if not res.stdout.strip():
            return set()
        out = pd.read_csv(StringIO(res.stdout), sep='\t', header=None)
        return set(out[3].astype(int))

    gene_idx  = intersect_row_indices(gene_bed_path)
    promo_idx = intersect_row_indices(promoter_bed_path)

    bm['feature_type'] = 'intergenic'
    bm.loc[bm['row_idx'].isin(gene_idx),  'feature_type'] = 'gene_body'
    bm.loc[bm['row_idx'].isin(promo_idx), 'feature_type'] = 'promoter'  # override

    return bm.drop(columns='row_idx')


# ── Apply to all available CpG samples ────────────────────────────────────────
feat_data = {}
for sample in available['CpG']:
    print(f'  Assigning features for {sample} ... ', end='', flush=True)
    feat_data[sample] = assign_feature_type(
        sample, gene_bed_path, promoter_bed_path, TMPDIR)
    vc = feat_data[sample]['feature_type'].value_counts()
    total = vc.sum()
    print('  '.join(f'{k}: {v:,} ({v/total*100:.1f}%)' for k, v in vc.items()))

# ── Summary table ─────────────────────────────────────────────────────────────
summary_rows = []
for sample, df in feat_data.items():
    for ft, grp in df.groupby('feature_type'):
        summary_rows.append({
            'sample': sample,
            'feature_type': ft,
            'n_sites': len(grp),
            'mean_meth': grp['pct_mod'].mean(),
            'median_meth': grp['pct_mod'].median(),
        })
feat_summary = pd.DataFrame(summary_rows)
print()
print(feat_summary.pivot_table(
    index='sample', columns='feature_type', values='mean_meth'
).round(2).to_string())


In [ ]:
# ── Violin plot — methylation by feature type ─────────────────────────────────
feat_plot_df = pd.concat(
    [df.assign(sample=s) for s, df in feat_data.items()],
    ignore_index=True,
)
# Subsample for speed: 50 000 random sites per (sample × feature_type) combo
rng = np.random.default_rng(42)
groups = feat_plot_df.groupby(['sample', 'feature_type'])
feat_sub = groups.apply(
    lambda g: g.sample(min(len(g), 50_000), random_state=42)
).reset_index(drop=True)

FT_ORDER     = ['promoter', 'gene_body', 'intergenic']
FT_LABELS    = {'promoter': 'Promoter\n(2 kb)', 'gene_body': 'Gene body',
                'intergenic': 'Intergenic'}
SAMPLE_ORDER = [s for s in SAMPLES if s in feat_data]
FT_PALETTE   = {'promoter': '#9C27B0', 'gene_body': '#FF9800', 'intergenic': '#607D8B'}

fig, axes = plt.subplots(1, len(SAMPLE_ORDER), figsize=(4 * len(SAMPLE_ORDER), 5),
                         sharey=True)
for ax, sample in zip(axes, SAMPLE_ORDER):
    sub = feat_sub[feat_sub['sample'] == sample]
    sns.violinplot(
        data=sub, x='feature_type', y='pct_mod', order=FT_ORDER,
        palette=FT_PALETTE, inner='quartile', linewidth=0.8, ax=ax,
    )
    ax.set_title(SAMPLE_META[sample]['label'])
    ax.set_xlabel('')
    ax.set_ylabel('CpG methylation (%)' if ax == axes[0] else '')
    ax.set_xticklabels([FT_LABELS[ft] for ft in FT_ORDER], fontsize=8)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    ax.set_ylim(-5, 105)
    sns.despine(ax=ax)

fig.suptitle('CpG methylation by genomic feature type', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig2_methylation_by_feature.pdf', bbox_inches='tight')
plt.show()


## Section 3 — PCA and Hierarchical Clustering of Samples

Builds a **site × sample methylation matrix** from the `CpG` bedMethyl files:

1. Load per-sample CpG methylation (min coverage ≥ 5)
2. Outer-join on `(chrom, start)` → retain sites present in **all** samples
3. Variance filter: keep the **top 50 000 most variable** CpG sites (sufficient for PCA / clustering)
4. PCA (sklearn) + hierarchical clustering heatmap (seaborn)

*Note: with only 4 samples, PCA and clustering are indicative rather than statistically definitive.*


In [ ]:
# ── Build CpG methylation % matrix ───────────────────────────────────────────
MIN_COV      = 5       # minimum coverage to include a site
TOP_N_SITES  = 50_000  # variance filter for PCA / clustering

cpg_samples = available['CpG']
frames = []
for sample in cpg_samples:
    fpath = RESULTS_DIR / 'CpG' / sample / f'{sample}.bedMethyl'
    print(f'  Loading {sample} ... ', end='', flush=True)
    bm = pd.read_csv(
        fpath, sep='\t', header=None, names=BEDMETHYL_COLS,
        usecols=[0, 1, 9, 10],
        dtype={'chrom': str, 'start': int, 'N_valid_cov': int, 'pct_mod': float},
    )
    bm.columns = ['chrom', 'start', 'N_valid_cov', 'pct_mod']
    bm = bm[bm['N_valid_cov'] >= MIN_COV]
    bm = bm.rename(columns={'pct_mod': sample})[['chrom', 'start', sample]]
    frames.append(bm)
    print(f'{len(bm):,} sites (cov ≥ {MIN_COV})')

# Inner join across all samples
from functools import reduce
matrix = reduce(
    lambda a, b: a.merge(b, on=['chrom', 'start'], how='inner'),
    frames,
)
print(f'\nCommon sites (all samples, cov ≥ {MIN_COV}): {len(matrix):,}')

# Variance filter
site_var = matrix[cpg_samples].var(axis=1)
top_sites = site_var.nlargest(min(TOP_N_SITES, len(matrix))).index
mat_filt  = matrix.loc[top_sites, cpg_samples]
print(f'Top {len(mat_filt):,} most variable sites retained for PCA / clustering')

# Matrix for PCA: shape = (samples × sites)
X = mat_filt.T.values          # rows = samples, cols = sites
print(f'Matrix shape (samples × sites): {X.shape}')


In [ ]:
# ── PCA ───────────────────────────────────────────────────────────────────────
pca   = PCA(n_components=min(len(cpg_samples), 3))
X_sc  = StandardScaler().fit_transform(X)
X_pca = pca.fit_transform(X_sc)

var_explained = pca.explained_variance_ratio_ * 100
print('Explained variance:',
      '  '.join(f'PC{i+1}: {v:.1f}%' for i, v in enumerate(var_explained)))

fig, ax = plt.subplots(figsize=(6, 5))
for i, sample in enumerate(cpg_samples):
    ax.scatter(X_pca[i, 0], X_pca[i, 1],
               color=SAMPLE_META[sample]['color'], s=160, zorder=3,
               edgecolors='white', linewidth=0.8)
    ax.annotate(SAMPLE_META[sample]['label'], (X_pca[i, 0], X_pca[i, 1]),
                textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.set_xlabel(f'PC1 ({var_explained[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({var_explained[1]:.1f}%)' if pca.n_components_ > 1 else 'PC2')
ax.set_title(f'PCA of CpG methylation\n(top {len(mat_filt):,} variable sites, cov ≥ {MIN_COV})')
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig3a_pca.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── Hierarchical clustering heatmap ──────────────────────────────────────────
# Subsample to 10 000 sites for the heatmap
rng       = np.random.default_rng(0)
hm_idx    = rng.choice(len(mat_filt), min(10_000, len(mat_filt)), replace=False)
hm_mat    = mat_filt.iloc[hm_idx]          # (10 000 sites × samples)

sample_colors = pd.Series(
    [SAMPLE_META[s]['color'] for s in hm_mat.columns],
    index=hm_mat.columns,
)

cg = sns.clustermap(
    hm_mat.T,                              # (samples × sites) for heatmap as rows = samples
    col_cluster=True,
    row_cluster=(len(cpg_samples) > 2),    # clustering rows (samples) needs ≥ 3
    cmap='RdBu_r',
    center=50,
    vmin=0, vmax=100,
    xticklabels=False,
    yticklabels=[SAMPLE_META[s]['label'] for s in hm_mat.columns],
    figsize=(12, max(2 * len(cpg_samples), 5)),
    cbar_kws={'label': 'CpG methylation (%)'},
)
cg.fig.suptitle(
    f'Hierarchical clustering — CpG methylation\n'
    f'({len(hm_idx):,} random sites from top {len(mat_filt):,} variable)',
    y=1.02,
)
plt.savefig(OUT_DIR / 'fig3b_clustermap.pdf', bbox_inches='tight')
plt.show()


## Section 4 — Coverage and Methylation Distribution (ONT QC)

For ONT native methylation, conventional bisulfite **M-bias** (read-position artefact) does not apply.  
Instead, we inspect four complementary QC diagnostics using the `allC` bedMethyl:

| Plot | What to check |
|------|--------------|
| Coverage histogram | Distribution of `N_valid_cov`; flag unusually low/high coverage |
| Per-chromosome median coverage | Detect poorly aligned or absent chromosomes |
| Methylation % histogram | Bimodal distribution (hypomethylated vs methylated) expected |
| pct_mod vs N_valid_cov | Low-coverage sites may have extreme methylation estimates |


In [ ]:
# ── Coverage distribution ─────────────────────────────────────────────────────
allC_samples = list(allC_data.keys())
SAMPLE_ORDER = [s for s in SAMPLES if s in allC_data]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: coverage histogram (clipped at 200x for readability)
ax = axes[0]
for sample in SAMPLE_ORDER:
    cov = allC_data[sample]['N_valid_cov'].clip(upper=200)
    ax.hist(cov, bins=80, alpha=0.5, label=SAMPLE_META[sample]['label'],
            color=SAMPLE_META[sample]['color'], edgecolor='none')
ax.set_xlabel('Coverage (N_valid_cov, clipped at 200)')
ax.set_ylabel('Number of cytosine positions')
ax.set_title('Coverage distribution (allC)')
ax.legend(frameon=False, fontsize=8)
sns.despine(ax=ax)

# Right: per-chromosome median coverage (box per sample)
ax = axes[1]
chrom_cov = []
for sample in SAMPLE_ORDER:
    by_chr = allC_data[sample].groupby('chrom')['N_valid_cov'].median().reset_index()
    by_chr['sample'] = sample
    chrom_cov.append(by_chr)
chrom_cov_df = pd.concat(chrom_cov, ignore_index=True)

# Keep only main chromosomes (NC_...) for clarity
main_chr = chrom_cov_df['chrom'].str.startswith('NC_')
chrom_cov_df = chrom_cov_df[main_chr]

sns.boxplot(data=chrom_cov_df, x='sample', y='N_valid_cov',
            order=SAMPLE_ORDER,
            palette=[SAMPLE_META[s]['color'] for s in SAMPLE_ORDER],
            width=0.5, linewidth=0.8, ax=ax)
ax.set_xticklabels([SAMPLE_META[s]['label'] for s in SAMPLE_ORDER])
ax.set_xlabel('')
ax.set_ylabel('Median coverage per chromosome')
ax.set_title('Per-chromosome median coverage (main chromosomes)')
sns.despine(ax=ax)

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig4a_coverage_distribution.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── Methylation % distribution ────────────────────────────────────────────────
# Separate panels for each context (CG / CHG / CHH) — use classified allC data
ctx_order   = ['CG', 'CHG', 'CHH']
n_ctx       = len(ctx_order)
n_samples   = len(SAMPLE_ORDER)

fig, axes = plt.subplots(n_ctx, n_samples,
                         figsize=(4 * n_samples, 3 * n_ctx),
                         sharex=True, sharey='row')
if n_samples == 1:
    axes = axes.reshape(-1, 1)

for row_i, ctx in enumerate(ctx_order):
    for col_j, sample in enumerate(SAMPLE_ORDER):
        ax  = axes[row_i][col_j]
        if sample not in allC_ctx:
            ax.set_visible(False)
            continue
        sub = allC_ctx[sample][allC_ctx[sample]['context'] == ctx]
        ax.hist(sub['pct_mod'], bins=50, color=SAMPLE_META[sample]['color'],
                edgecolor='none', alpha=0.8)
        ax.set_title(f'{SAMPLE_META[sample]["label"]} — {ctx}', fontsize=9)
        if col_j == 0:
            ax.set_ylabel('Count')
        if row_i == n_ctx - 1:
            ax.set_xlabel('Methylation (%)')
        ax.xaxis.set_major_formatter(mticker.PercentFormatter())
        sns.despine(ax=ax)

fig.suptitle('Methylation % distribution by context and sample (allC)', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig4b_methylation_distribution.pdf', bbox_inches='tight')
plt.show()


In [ ]:
# ── Hexbin: methylation % vs coverage ────────────────────────────────────────
# Helps detect whether low-coverage sites have extreme methylation estimates
fig, axes = plt.subplots(1, len(SAMPLE_ORDER), figsize=(5 * len(SAMPLE_ORDER), 4),
                         sharey=True)
if len(SAMPLE_ORDER) == 1:
    axes = [axes]

for ax, sample in zip(axes, SAMPLE_ORDER):
    df = allC_data[sample]
    cov_clip = df['N_valid_cov'].clip(upper=150)
    hb = ax.hexbin(cov_clip, df['pct_mod'], gridsize=50, cmap='YlOrRd',
                   mincnt=1, bins='log')
    ax.set_xlabel('Coverage (clipped at 150×)')
    ax.set_ylabel('Methylation (%)' if ax == axes[0] else '')
    ax.set_title(SAMPLE_META[sample]['label'])
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())
    plt.colorbar(hb, ax=ax, label='log10(count)')
    sns.despine(ax=ax)

fig.suptitle('Methylation % vs coverage — hexbin (allC, all contexts)',
             y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig4c_methylation_vs_coverage_hexbin.pdf', bbox_inches='tight')
plt.show()
